In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import requests

import pandas as pd

from time import sleep

import datetime

from bs4 import BeautifulSoup

from pandas import ExcelWriter

import os

    

In [42]:
# %%
#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'CA OSFI' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.2")

now=datetime.datetime.now()

filename = '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)





Running CA OSFI Web Scraping Tool v.2


In [43]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_argument("--disable-search-engine-choice-screen")

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

actionChains = ActionChains(driver)





In [44]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

    'CA OSFI 1': 'Domestic Banks', 

	'CA OSFI 2': 'Foreign Banks', 

	'CA OSFI 3': 'Foreign Bank Branches - Full Service', 

	'CA OSFI 4': 'Foreign Bank Branches - Lending', 

	'CA OSFI 5': 'Trust Companies', 

	'CA OSFI 6': 'Loan Companies', 

	'CA OSFI 9': 'Canadian Life Insurance Companies', 

	'CA OSFI 10': 'Foreign Life Insurance Companies', 

	'CA OSFI 11': 'Canadian Fraternal Benefit Societies', 

	'CA OSFI 12': 'Foreign Fraternal Benefit Societies', 

	'CA OSFI 13': 'Canadian Property & Casualty Insurance Companies', 

	'CA OSFI 14': 'Foreign Property & Casualty Insurance Companies', 

	'CA OSFI 15': 'Foreign Bank Representative Offices',

	'CA OSFI 16': 'Canadian Mortgage Insurers', 

    }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}



processdate = now.strftime('%Y-%m-%d')




    

In [45]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict




b'{"help": "https://open.canada.ca/data/en/api/3/action/help_show?name=datastore_search", "success": true, "result": {"include_total": true, "limit": 100, "records_format": "objects", "resource_id": "2282e8c7-f949-454d-9c34-b6a78d94b162", "total_estimation_threshold": null, "records": [{"_id":2021,"Plan Registration Number":"56752","Plan Name":"ADM Pension Plan for Hourly Employees - Canada","Plan Sponsor Name":"ADM Agri Industries Company","Administrator Contact Name":"Ms. Tracy Hood","Address":"4666 FARIES PKW,","City":"DECATUR","Province State":"IL"},{"_id":2022,"Plan Registration Number":"55103","Plan Name":"ADM Pension Plan for Salaried Employees - Canada","Plan Sponsor Name":"ADM Agri Industries Company","Administrator Contact Name":"Ms. Tracy Hood","Address":"4666 FARIES PARKWAY,","City":"DECATUR","Province State":"IL"},{"_id":2023,"Plan Registration Number":"57518","Plan Name":"AT&T Global Services Canada Co. Retirement Plan","Plan Sponsor Name":"AT&T Global Services Canada Co.

In [ ]:

resource_id = '945045fa-2de0-47d4-aad2-144d69467824'
base_url = 'https://open.canada.ca/data/en/api/3/action/datastore_search'
all_records = []
start = 0

while True:
    params = {
        'resource_id': resource_id,
        'limit': 1000,  # max allowed per page
        'offset': start
    }
    response = requests.get(base_url, params=params)
    data = response.json()

    if not data['success']:
        print("API request failed.")
        break

    records = data['result']['records']
    all_records.extend(records)

    if len(records) < 1000:
        break  # no more pages

    start += 1000

print(f"Total records retrieved: {len(all_records)}")

for records in all_records:
    #print(records)
    # print(records['FI Group Name'])
    # print(records['FI Industry Name'])
    # print(records['Address Line 1'])
    # print(records['Address Line 2'])
    # print(records['City'])
    # print(records['Province State'])
    # print(records['Postal ZIP Code'])

    for key, value in regdict.items():
        if records['FI Industry Name'] == value:
            sqldict['ListCode'].append(key.split(' ')[-1])


    sqldict['Name'].append(records['Company Name'])
    sqldict['ListName'].append(records['FI Industry Name'])
    sqldict['Typology'].append(records['FI Group Name'])
    sqldict['Address_1'].append(records['Address Line 1'])
    sqldict['Address_2'].append(records['Address Line 2'])
    sqldict['City'].append(records['City'])
    sqldict['Zip'].append(records['Postal ZIP Code'])
    sqldict['RegulationType'].append('Regulated')
    sqldict['ListProcessDate'].append(processdate)
    sqldict['RegCtry'].append('CA')
    sqldict['RegCode'].append('OSFI')

    sqldict = bourange_same_length_array(sqldict)


Total records retrieved: 359


In [47]:

# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_25488\2733503114.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [49]:
df.to_csv('total_csv.csv')